# 01 — Explore an AOI (Phase 1)

Survey the satellite coverage available for an area of interest, export a few
scenes, and look at them. Nothing here is a detection result: this notebook
only acquires data and inspects it.

> **Research prototype output. Not an official warning. Verify with authoritative sources.**

**Prerequisites:** the `himalaya` conda environment, `.env` with `GEE_PROJECT`,
and `earthengine authenticate` run once. See the README.

To explore a different area, edit `aoi` in `config/config.yaml` — nothing in
this notebook or the code paths assumes a particular site.

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Run from anywhere: put the repository root on the path.
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from config.loader import load_config, config_hash, ensure_output_dirs, resolved_paths
from utils.logging import setup_logging, new_run_id

cfg = load_config()
paths = ensure_output_dirs(cfg)
run_id = new_run_id()
setup_logging(cfg, run_id=run_id)
chash = config_hash(cfg)

print(f"AOI        : {cfg.aoi.name}")
print(f"source     : {cfg.aoi.source}")
print(f"window     : {cfg.dates.query_window.start} to {cfg.dates.query_window.end}")
print(f"run_id     : {run_id}")
print(f"config_hash: {chash[:16]}...")

## 1. Projection

The UTM zone is **derived** from the AOI centroid, never assumed (brief,
Section 6). Every area and volume downstream is computed in this CRS.

In [ ]:
from acquisition.gee_client import aoi_area_km2, aoi_to_ee_geometry, initialize, resolve_export_crs

initialize(cfg)
geom = aoi_to_ee_geometry(cfg.aoi)

print(f"centroid    : {cfg.aoi_centroid}")
print(f"export CRS  : {resolve_export_crs(cfg)}")
print(f"AOI area    : {aoi_area_km2(geom):.3f} km^2")
print("\nNote: exports cover the AOI's *bounding box*, which circumscribes a")
print("point+buffer AOI, so the raster extent is larger than the area above.")

## 2. What coverage exists?

Listing downloads no pixels. Cloud fraction is reported per scene so the choice
of what to export is an informed one — and so a cloudy window is visibly a
cloudy window rather than an apparent absence of data.

In [ ]:
from acquisition import landsat, sentinel1, sentinel2

print("=== Sentinel-2 ===")
for s in sentinel2.list_scenes(cfg):
    print(f"  {s.acquisition_start:%Y-%m-%d}  tile {s.orbit_or_tile}  cloud {s.cloud_fraction:6.2f}%  {s.platform}")

print("\n=== Sentinel-1 (grouped by orbit pass; they must stay separate) ===")
for direction, scenes in sorted(sentinel1.list_scenes_by_pass(cfg).items()):
    print(f"  {direction}: {len(scenes)} scene(s)")
    for s in scenes[:5]:
        print(f"    {s.acquisition_start:%Y-%m-%d}  {s.orbit_or_tile}  rel.orbit {s.relative_orbit}")

print("\n=== Landsat 8/9 ===")
for s in landsat.list_scenes(cfg):
    print(f"  {s.acquisition_start:%Y-%m-%d}  {s.orbit_or_tile}  cloud {s.cloud_fraction:6.2f}%  {s.platform}")

## 3. The DEM epoch — the Branch A/B hinge

This is the single most consequential number for Phase 10. A lake first
observed **after** the hinge date can have a depth estimated from the DEM
(Branch A). A lake that already existed means the DEM recorded its *water
surface*, the bed is unknown, and depth must be reported `unavailable`
(Branch B). It is measured from the catalog here, never assumed.

In [ ]:
from acquisition import dem

epoch = dem.acquisition_epoch(cfg)
print(epoch.summary())
print(f"\nhinge date: {epoch.branch_hinge_date}")
print(f"tile versions present: {epoch.tile_versions_present} -> using {epoch.tile_version_selected}")
print(f"entries: {epoch.entries_selected} of {epoch.entries_intersecting_aoi} after de-duplication")

## 4. Export

Each GeoTIFF is written with a `.json` metadata sidecar carrying sensor,
acquisition dates, orbit/tile, processing level and source collection (brief,
Section 5), plus the real raster dimensions read back from disk.

Keep `limit` small: these are real downloads.

In [ ]:
exports = []
exports += dem.export_dem(cfg, run_id=run_id, config_hash=chash)[0]
exports += sentinel2.export_scenes(cfg, limit=1, run_id=run_id, config_hash=chash)
exports += sentinel1.export_scenes(cfg, limit_per_pass=1, run_id=run_id, config_hash=chash)
exports += landsat.export_scenes(cfg, limit=1, run_id=run_id, config_hash=chash)

for raster, sidecar in exports:
    print(f"  {raster.relative_to(paths['raw'])}")

## 5. Inspect what landed

Open every exported raster and report its real geometry and value ranges. This
is the Phase 1 gate check: a file that does not open here will not open in QGIS.

In [ ]:
import rasterio

for path in sorted(paths["raw"].rglob("*.tif")):
    with rasterio.open(path) as src:
        tags = src.tags()
        print(f"{path.relative_to(paths['raw'])}")
        print(f"   {src.width}x{src.height} | {src.count} band(s) {src.descriptions}")
        print(f"   {src.dtypes[0]} | {src.crs} | {src.res[0]:.0f} m")
        print(f"   {tags.get('HDM_SENSOR')} / {tags.get('HDM_PROCESSING_LEVEL')} / {tags.get('HDM_ORBIT_OR_TILE')}\n")

## 6. Quick-look figures

Written to `outputs/figures/`. These are for eyeballing the acquisition, not
for analysis.

Watch the SCL panel: the extent of the `dark/shadow` class is the problem the
whole project exists to solve, and the Sentinel-1 panel shows why ascending and
descending passes cannot be mixed in steep terrain.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def stretch(a, lo=2, hi=98):
    v = a[np.isfinite(a)]
    p1, p2 = np.percentile(v, [lo, hi])
    return np.clip((a - p1) / max(p2 - p1, 1e-9), 0, 1)


s2_refl = sorted(paths["raw"].glob("sentinel2/*_refl.tif"))
dem_tif = sorted(paths["raw"].glob("dem/*.tif"))

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

if s2_refl:
    with rasterio.open(s2_refl[0]) as s:
        b = {n: s.read(i).astype(float) for i, n in enumerate(s.descriptions, 1)}
    axes[0].imshow(np.dstack([stretch(b["B4"]), stretch(b["B3"]), stretch(b["B2"])]))
    axes[0].set_title(f"S2 true colour\n{s2_refl[0].name}", fontsize=9)
    axes[1].imshow(np.dstack([stretch(b["B12"]), stretch(b["B8"]), stretch(b["B3"])]))
    axes[1].set_title("S2 false colour (B12/B8/B3)", fontsize=9)

if dem_tif:
    with rasterio.open(dem_tif[0]) as s:
        elev = s.read(1).astype(float)
    dy, dx = np.gradient(elev, cfg.gee.default_scale_m)
    slope, aspect = np.arctan(np.hypot(dx, dy)), np.arctan2(-dx, dy)
    az, alt = np.radians(315.0), np.radians(45.0)
    hs = np.sin(alt) * np.cos(slope) + np.cos(alt) * np.sin(slope) * np.cos(az - aspect)
    axes[2].imshow(hs, cmap="gray")
    axes[2].set_title(f"DEM hillshade\n{elev.min():.0f}-{elev.max():.0f} m", fontsize=9)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f"{cfg.aoi.name}   |   {cfg.project.disclaimer}", fontsize=10)
fig.tight_layout()
fig.savefig(paths["figures"] / "explore_aoi.png", dpi=110, bbox_inches="tight")
plt.show()

---

## Next

Phase 2 takes these exports and produces cloud masks, SAR dB/speckle handling,
**terrain**-shadow masks (kept strictly separate from cloud shadow), SAR
layover/shadow masks, and a co-registered 10 m stack.